In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import(
    accuracy_score,classification_report,confusion_matrix,ConfusionMatrixDisplay
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

POPULATION_SIZE = 15
MAX_ITER = 40
K_MIN,K_MAX = 1,15
N_FEATURES = 13

print(f"✅Library Imported Successfull")
print(f"Random seed  : {RANDOM_SEED}")
print(f"GA Population Size  : {POPULATION_SIZE}")
print(f"Max iterations: {MAX_ITER}")
print(f"K_Neighbour range : [{K_MIN}, {K_MAX}]")

In [ ]:
# Data Preprocessing

# X_train = Training data
# y_train = Output for Training data
# X_test = Testing data
# y_test = Output for Testing data

def load_data(test_size: float=0.3, random_state: int = RANDOM_SEED):
    wine = load_wine()
    X, y = wine.data, wine.target
    feature_names = list(wine.feature_names)
    class_names   = list(wine.target_names)

    X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=test_size,random_state=random_state,stratify=y)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    # This is used to Standarized the data so that each data is in the same range
    # Ex: 1st feature range = {1,100}, 2nd feature range = {10000,100000} then it 
    # standardized both and make then in the same range like {-1 to 1}

    return X_train, X_test, y_train, y_test, feature_names, class_names

X_train, X_test, y_train, y_test, FEATURE_NAMES, CLASS_NAMES = load_data(test_size=0.3)

print("Total Sample: 178")
print(f"Training Set: {X_train.shape[0]} Samples, {X_train.shape[1]} Features")
print(f"Testing Set: {X_test.shape[0]} Sampeles, {X_test.shape[1]} Features")
print(f"Classes Names: {CLASS_NAMES}")
print(f"FEATURE_NAMES: {FEATURE_NAMES}")

In [ ]:
# Baseline KNN Model

# X_train = Input Data
# y_train = output data for input
def train_knn(X_train, y_train, k:int = 5, feature_mask: np.ndarray = None)-> KNeighborsClassifier:
    if feature_mask is not None:
        mask = feature_mask.astype(bool)
        if mask.sum()==0:
            mask[0]=True
        X_train = X_train[:,mask]
        
    classifier = KNeighborsClassifier(n_neighbors=int(k))
    classifier.fit(X_train,y_train)    # Learn on input and output
    return classifier

def evaluate_model(classifier: KNeighborsClassifier, X_test,y_test,feature_mask: np.ndarray = None)->float:
    # X_test = Testing data
    # y_test = output for Testing data

    if feature_mask is not None:
        mask = feature_mask.astype(bool)
        if mask.sum()==0:
            mask[0]=True
        X_test = X_test[:,mask]
    
    prediction = classifier.predict(X_test)
    prediction_accuracy = accuracy_score(y_test,prediction)
    return prediction_accuracy

def fitness_function(solution:np.ndarray, X_train, y_train, X_val, y_val)->float:
    # Solution is an array that consist [K,f1,f2,f3...]
    # K is the value for K-neighbour and the remaining is the value for feature_mask
    
    k = int(np.clip(round(solution[0]),K_MIN,K_MAX))
    feature_mask = (solution[1:]>=0.5).astype(int)

    trained_model = train_knn(X_train,y_train,k,feature_mask)
    model_accuracy = evaluate_model(trained_model,X_val,y_val,feature_mask)
    return model_accuracy


baseline_model = train_knn(X_train,y_train,k=5,feature_mask=None)
baseline_accuracy = evaluate_model(baseline_model,X_test,y_test,feature_mask=None)

cross_validation_score = cross_val_score(KNeighborsClassifier(n_neighbors=5), np.vstack([X_train, X_test]), np.concatenate([y_train, y_test]), cv=5)
# Mean of Accuracy on 5 Different validation

print("BASELINE KNN RESULT")
print(f"-"*50)
print(f"K (Neighbour): 5")
print(f"Feature Used: All {N_FEATURES}")
print(f"Baseline Accuracy: {baseline_accuracy:.4f} ({baseline_accuracy*100:.2f}%)")
print(f"5 Fold CV Accuracy: {cross_validation_score.mean():.4f} ± {cross_validation_score.std():.4f}")

In [ ]:
# Whale Optimization Model (From Scratch)

def whale_optimization(X_train,y_train,x_val,y_val,population_size: int=POPULATION_SIZE, max_iteration: int=MAX_ITER, random_state: int=RANDOM_SEED):

    rng = np.random.default_rng(random_state)
    dim = N_FEATURES+1

    lb = np.array([K_MIN] + [0.0]*N_FEATURES)
    ub = np.array([K_MAX] + [1.0]*N_FEATURES)

    position = lb + rng.random((population_size,dim)) * (ub-lb)
    # Assigning all whales a random soluton in the search space
    # lb is used to shift all the whale into right search space and ub is 
    # used to all the paramter comes into the range

    initial_fitness = np.array([fitness_function(position[i],X_train,y_train,X_test,y_test) for i in range(population_size)])
    # Initial Fitness Function for the whales before Optimization

    best_idx = np.argmax(initial_fitness)
    best_position = position[best_idx].copy()
    best_fitness_score = initial_fitness[best_idx]
    # Best postion and fitness score out of all the whales

    convergence = [best_fitness_score]

    # Whale Optimization Algorithm
    for t in range(max_iteration):
        None